In [12]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(model = "gemma-4-31b-it",temperature= 0.2)


In [13]:
with open("credit_card_expenses.txt", "r") as f:
    credit_card_analyzer = f.read()

print(credit_card_analyzer[:100])    

05/02 Supermarket Groceries $112.40
05/03 Apple Services Recurring $9.99
05/04 DoorDash Food Deliver


In [14]:
extraction_prompt = f"""
You are a certified forensic financial auditor.

Analyze the statement transactions below. For every transaction, extract:
1. Date (if present)
2. Merchant / Description
3. Estimated Category (Essentials, Dining/Discretionary, Subscriptions, Bank Fees)
4. Amount
5. Risk / Leak Tag:
   - FLAG AS 'DORMANT_LEAK' if it looks like a recurring subscription or fitness club.
   - FLAG AS 'SURGE' if it is an unusually high discretionary or dining expense.
   - FLAG AS 'FEE_PENALTY' for late fees, overdrafts, interest charges, or card maintenance fees.
   - Otherwise mark as 'NORMAL'.

Format your response as a clean bullet list:
- [Category] Merchant: $Amount | Tag: DORMANT_LEAK / SURGE / FEE_PENALTY / NORMAL | Notes

Statement Data:
{credit_card_analyzer}
"""

extraction_response = llm.invoke(extraction_prompt)
extracted_values = extraction_response.text

print("=== STAGE 1: EXTRACTED VALUES ===")
print(extracted_values)

=== STAGE 1: EXTRACTED VALUES ===
- [Essentials] Supermarket Groceries: $112.40 | Tag: NORMAL | Standard grocery expenditure.
- [Subscriptions] Apple Services Recurring: $9.99 | Tag: DORMANT_LEAK | Recurring digital subscription.
- [Dining/Discretionary] DoorDash Food Delivery: $48.20 | Tag: NORMAL | Standard food delivery.
- [Subscriptions] Netflix 4K UHD: $22.99 | Tag: DORMANT_LEAK | Recurring streaming service.
- [Dining/Discretionary] Uber Rides: $31.50 | Tag: NORMAL | Transportation expense.
- [Subscriptions] Anytime Fitness Gym: $64.00 | Tag: DORMANT_LEAK | Recurring fitness membership.
- [Subscriptions] Spotify Family: $16.99 | Tag: DORMANT_LEAK | Recurring music subscription.
- [Bank Fees] Late Payment Penalty Fee: $39.00 | Tag: FEE_PENALTY | Penalty for missed payment deadline.
- [Dining/Discretionary] DoorDash Food Delivery: $52.10 | Tag: NORMAL | Standard food delivery.
- [Essentials] Electricity Utility Bill: $145.00 | Tag: NORMAL | Essential utility payment.
- [Subscriptio

In [15]:
budget_prompt = f"""
You are an expert personal financial advisor and debt counselor.

Based on the flagged audit below, generate two clearly designated sections:

SECTION 1 - FINANCIAL HEALTH DIAGNOSIS:
Provide a 4-5 line assessment explaining the user's biggest spending leaks, 
the total estimated waste on avoidable fees/subscriptions, and overall financial risk.

SECTION 2 - CUT-LIST & RESTRUCTURING PLAN:
1. Immediate Cut-List: Name specific subscriptions or repeated fees that should be canceled or disputed immediately.
2. 50/30/20 Action Target: Briefly indicate what adjustments are required to align their cash flow with 50% Needs, 30% Wants, and 20% Savings/Debt Repayment.

Flagged Audit Data:
{extracted_values}
"""

budget_response = llm.invoke(budget_prompt)

print("=== STAGE 2: Budget PLAN ===")
print(budget_response.text)

=== STAGE 2: Budget PLAN ===
**SECTION 1 - FINANCIAL HEALTH DIAGNOSIS:**
Your primary spending leaks are "dormant" subscriptions and avoidable banking penalties, totaling **$209.76 in immediate waste**. The presence of late fees and interest charges is a critical red flag, indicating a breakdown in cash flow management or an escalating debt cycle. Your discretionary spending—specifically high-end dining and delivery—is disproportionately high compared to your essential costs. Overall, you are at moderate-to-high financial risk due to inefficient capital leakage and poor debt servicing.

**SECTION 2 - CUT-LIST & RESTRUCTURING PLAN:**

**1. Immediate Cut-List:**
*   **Cancel Immediately (Dormant Leaks):** Anytime Fitness ($64.00), Netflix 4K ($22.99), Spotify Family ($16.99), Amazon Prime Monthly ($14.99), and Apple Services ($9.99).
*   **Dispute/Address:** Contact your bank to request a one-time courtesy waiver for the Late Payment Penalty Fee ($39.00).
*   **Behavioral Cut:** Eliminat